# Talkie 13B encoder — smoke test

Verifies, on an A100 (>=28 GB VRAM, bf16), that we can:
1. load the Talkie 1930 base model,
2. extract full-sequence hidden states (their `forward` only returns last-position logits),
3. pool them into sentence embeddings,
4. see a first sign that the embeddings carry **register** — period sentences should sit closer to each other than to modern paraphrases.

Run before committing to the full 200k-question pass. Self-contained: no repo checkout needed.

In [ ]:
# 0. Check we actually have an A100-class GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No CUDA GPU — Talkie 13B needs an A100 (>=28GB)'
print('CUDA:', torch.cuda.get_device_name(0))

In [ ]:
# 1. Install Talkie (downloads ~53GB weights on first model load)
!pip -q install git+https://github.com/talkie-lm/talkie

In [ ]:
# 2. Load the base model (weights download on first run)
from talkie import Talkie
t = Talkie('talkie-1930-13b-base')
model, tok = t.model, t.tokenizer
# sanity: the submodules our readout relies on must exist
for attr in ('embed', 'blocks', 'cos', 'sin'):
    assert hasattr(model, attr), f'model has no .{attr} — check Talkie version'
print('layers:', len(model.blocks), ' device:', next(model.parameters()).device)

In [ ]:
# 3. Full-sequence hidden states = Talkie's forward WITHOUT the last-position slice + head.
#    (mirrors src/talkie/model.py forward; returns [B, T, H])
import torch, torch.nn.functional as F, numpy as np

@torch.no_grad()
def hidden_states(model, input_ids):
    _, seq_len = input_ids.shape
    cos_sin = model.cos[:, :seq_len], model.sin[:, :seq_len]
    x = model.embed(input_ids)
    x = F.rms_norm(x, (x.shape[-1],))
    e_x = x
    for block in model.blocks:
        x = block(e_x, x, cos_sin)
    return F.rms_norm(x, (x.shape[-1],))

@torch.no_grad()
def encode(texts, max_len=64, pooling='mean'):
    device = next(model.parameters()).device
    chunk = [ (tok.encode(s, allowed_special='all')[:max_len] or [0]) for s in texts ]
    T = max(len(ids) for ids in chunk)
    x = torch.zeros(len(chunk), T, dtype=torch.long, device=device)
    mask = torch.zeros(len(chunk), T, dtype=torch.bool, device=device)
    for r, ids in enumerate(chunk):
        x[r, :len(ids)] = torch.tensor(ids, device=device)
        mask[r, :len(ids)] = True
    h = hidden_states(model, x).float()
    if pooling == 'mean':
        m = mask[..., None].float()
        v = (h * m).sum(1) / m.sum(1).clamp(min=1)
    else:
        v = h[torch.arange(h.size(0)), mask.sum(1) - 1]
    return v.cpu().numpy()

In [ ]:
# 4. Smoke test: 3 period questions vs 3 modern paraphrases of the same facts
period = [
    'Why are they called trade winds?',
    'Of what use is the gastric juice?',
    'What countries had glass windows first?',
]
modern = [
    "What's the deal with trade winds, exactly?",
    'Can you give me the TL;DR on what gastric juice does?',
    'Which countries were first to have glass windows, historically speaking?',
]
emb = encode(period + modern)
print('embedding shape:', emb.shape)   # (6, H) — H ~ 5120
assert emb.shape[0] == 6 and emb.ndim == 2

In [ ]:
# 5. Does the space carry register? period<->period should be closer than period<->modern.
def cos(a, b):
    a = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return a @ b.T

P, M = emb[:3], emb[3:]
pp = cos(P, P)[np.triu_indices(3, k=1)].mean()   # within period
pm = cos(P, M).mean()                             # period vs modern
print(f'mean cosine  period-period: {pp:.3f}   period-modern: {pm:.3f}')
print('register signal present (pp > pm)?', bool(pp > pm))

**Reading it:** the shape check confirms loading + hidden-state extraction work. `period-period > period-modern` is a first (tiny, n=3) sign the embeddings separate register — not proof, but a green light to run the labeled validation set. If they're equal, raw pooling may be too weak and we'd consider the LLM2Vec conversion.

Next: embed a labeled sample and measure real separation, then build the scorer on cached vectors.